In [ ]:
import geopandas as gpd

# 1. Last inn datasettene fra data-mappen
print("Laster inn data...")
tilfluktsrom = gpd.read_file('data/raw/Samfunnssikkerhet_42_Agder_25832_TilfluktsromOffentlige_GeoJSON.json')
flomsone = gpd.read_file('data/raw/Flomsone_200Aar.geojson')

# Sørg for at tilfluktsrom har riktig CRS registrert (filnavnet sier EPSG:25832)
if tilfluktsrom.crs is None:
    tilfluktsrom.set_crs(epsg=25832, inplace=True)

# 2. Gjør klar for romlig analyse (Spatial Join)
# Vi konverterer flomsonen til samme system som tilfluktsrommene (EPSG:25832) for nøyaktig meter-måling i Norge
flomsone = flomsone.to_crs(epsg=25832)

# Utfør analysen: Finn tilfluktsrom som krysser (intersects) flomsonen
print("Utfører romlig analyse...")
saarbare_tilfluktsrom = gpd.sjoin(tilfluktsrom, flomsone, how="inner", predicate="intersects")

# Skriv ut litt gøy statistikk til rapporten!
print(f"Totalt antall tilfluktsrom i Agder: {len(tilfluktsrom)}")
print(f"Antall sårbare tilfluktsrom i flomsonen: {len(saarbare_tilfluktsrom)}")
# Siden datasettet inneholder 'plasser', kan vi regne ut hvor mange som mister plassen sin!
print(f"Totalt antall tapte tilfluktsplasser ved 200-årsflom: {saarbare_tilfluktsrom['plasser'].sum()}")

# 3. Klargjør data for MapLibre Webkart (EPSG:4326)
print("Konverterer til Web Mercator (EPSG:4326)...")
flomsone_web = flomsone.to_crs(epsg=4326)
saarbare_web = saarbare_tilfluktsrom.to_crs(epsg=4326)
tilfluktsrom_web = tilfluktsrom.to_crs(epsg=4326) # Vi konverterer hele lista også, for sikkerhets skyld

# 4. Lagre de ferdige filene
flomsone_web.to_file('data/flomsone_web.geojson', driver='GeoJSON')
saarbare_web.to_file('data/saarbare_tilfluktsrom_web.geojson', driver='GeoJSON')
tilfluktsrom_web.to_file('data/tilfluktsrom_web.geojson', driver='GeoJSON')

print("Ferdig! Filene er klare for webkartet.")

Laster inn data...
Utfører romlig analyse...
Totalt antall tilfluktsrom i Agder: 65
Antall sårbare tilfluktsrom i flomsonen: 1
Totalt antall tapte tilfluktsplasser ved 200-årsflom: 150
Konverterer til Web Mercator (EPSG:4326)...
Ferdig! Filene er klare for webkartet.
